# Apply Naive Weight-only INT4 Quantization on Qwen3-8b

In [1]:
import tqdm
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from functools import partial
import gc
import os
os.environ['https_proxy'] = 'http://192.168.1.12:7891'

debug = True

if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

/root/workspace/qwen_cpu_deployment/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here we use wikitext-2 dataset for perplexity evaluation. The dataset is automatically downloaded by the code.

In [2]:
print("Loding wikitext datasets ...")
testenc = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', cache_dir="~/.cache/huggingface/datasets")
print("Done")

Loding wikitext datasets ...
Done


In [3]:
def evaluate(model, testenc, tokenizer):
    # we control the text length to avoid error posed by tiktoken
    testenc = tokenizer("\n\n".join(testenc['text']), return_tensors='pt')
    testenc = testenc.input_ids.to(model.device)
    nsamples = 40
    model = model.eval()

    nlls = []
    for i in tqdm.tqdm(range(nsamples), desc="evaluating Qwen on wikitext"):
        batch = testenc[:, (i * 1024):((i + 1) * 1024)].to(model.device)
        with torch.no_grad():
            lm_logits = model(batch).logits
        shift_logits = lm_logits[:, :-1, :].contiguous().float()
        shift_labels = testenc[:, (i * 1024):((i + 1) * 1024)][:, 1:]
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        neg_log_likelihood = loss.float() * 1024
        nlls.append(neg_log_likelihood)

    return torch.exp(torch.stack(nlls).sum() / (nsamples * 1024))

def get_model_size(model: nn.Module, data_width=16, group_size=-1):
    # store the quantization parameters: 1 fp16 scaling factors and 1 int4 zero point
    # this might not be precise
    if group_size != -1:
        data_width += (16 + 4) / group_size
    num_elements = 0
    for param in model.parameters():
        num_elements += param.numel()
    return num_elements * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

In [4]:
import lm_eval
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer

def evaluate_hf_model(model, tokenizer, batch_size="auto"):
    # Wrap the model with lm_eval's HFLM interface
    hf_model = HFLM(
        pretrained=model,
        tokenizer=tokenizer,
        # batch_size="auto",  # or an int like 4
        batch_size=batch_size,  # or an int like 4
        max_length=8192,
        dtype="float16"
    )

    # 2. Define the tasks you want to evaluate on
    task_names = [
        # "ceval-valid",
        "gsm8k"
    ]

    # 3. Run evaluation
    results = lm_eval.simple_evaluate(
        model=hf_model,
        tasks=task_names,
        log_samples=False,
        verbosity="INFO"
    )

    # 4. Print and save results
    print("Results:", results["results"])

2025-05-30 11:52:49,823	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 05-30 11:52:49 [__init__.py:243] Automatically detected platform cuda.


# Evaluate the performance of FP32 Qwen
## PPL

In [5]:

model_name = "Qwen/Qwen3-8B"

In [3]:

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)

Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.56it/s]


In [ ]:
fp32_perplexity = evaluate(model, testenc, tokenizer)
print(f"\nmodel perplexity: {fp32_perplexity:.2f}")


In [ ]:
model_size = get_model_size(model, data_width=32, group_size=-1)
print(f"model size: {model_size/MiB:.2f} MiB")

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

## GSM8k

In [ ]:
evaluate_hf_model(model, tokenizer)

2025-05-30 10:25:18,536	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 05-30 10:25:18 [__init__.py:243] Automatically detected platform cuda.


`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
2025-05-30:10:25:19 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:10:25:19 INFO     [evaluator:239] Using pre-initialized model
2025-05-30:10:25:31 INFO     [evaluator:286] gsm8k: Using gen_kwargs: {'until': ['Question:', '</s>', '<|im_end|>'], 'do_sample': False, 'temperature': 0.0}
2025-05-30:10:25:31 INFO     [api.task:434] Building contexts for gsm8k on rank 0...
100%|██████████| 1319/1319 [00:02<00:00, 471.17it/s]
2025-05-30:10:25:34 INFO     [evaluator:559] Running generate_until requests
Running generate_until requests: 100%|██████████| 1319/1

Results: {'gsm8k': {'alias': 'gsm8k', 'exact_match,strict-match': 0.8756633813495072, 'exact_match_stderr,strict-match': 0.009088880962028473, 'exact_match,flexible-extract': 0.8817285822592873, 'exact_match_stderr,flexible-extract': 0.008895075852434955}}


In [ ]:
!lm-eval --tasks gsm8k --model vllm --model_args pretrained=Qwen/Qwen3-8B,max_model_len=8192,dtype=float16 --batch_size auto --trust_remote_code

# Evaluate Performance Of Int4 Weight-quantized Model
Apply pseudo quantization to check the performance of quantized model directly

In [6]:
class QuantizedLinear(nn.Module):
    def __init__(self, 
                 dq_w,
                 w_bias,
                 activation_bitwidth=8, weight_bitwidth=4, q_method="A80W40"):
        super().__init__()
        # current version Pytorch does not support IntTensor as nn.Parameter

        self.activation_bitwidth = activation_bitwidth
        self.weight_bitwidth = weight_bitwidth
        self.q_blk_size = 32
        self.register_buffer(
            "dq_weight", dq_w
        )
        self.register_buffer(
            "weight_bias", w_bias
        )
        self.q_method = q_method

    def forward(self, x):
        # quantize activation
        assert self.weight_bitwidth == 4
        assert self.activation_bitwidth == 8
        ori_shape = x.shape
        x = x.reshape(-1, self.q_blk_size)
        x_blk_max_abs = (x.abs().max(dim=-1, keepdims=True)[0])
        x_blk_scale = x_blk_max_abs / (2 ** ( self.activation_bitwidth - 1 ) - 1)
        is_zero = x_blk_scale == 0
        x_blk_r_s = 1 / x_blk_scale
        x_blk_r_s[is_zero] = 0
        x_q = (x * x_blk_r_s).round()
        assert not (x_q.isnan().any())
        # apply fake quantization to simulate quantization error
        x_dq = (x_q) * x_blk_scale
        avg_error = (x- x_dq).abs().mean()
        # print(f"Average quantization error between x and x_dq: {avg_error.item()}")
        x_dq = x_dq.reshape(*ori_shape)
        
        
        w_dq = self.dq_weight
        
        output = torch.nn.functional.linear(
            x_dq,
            w_dq,
            self.weight_bias
        )
        return output

def pseudo_quantize_tensor_q4_zero_point(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_val = torch.amax(w, dim=-1, keepdim=True)
    min_val = torch.amin(w, dim=-1, keepdim=True)
    max_int = 2 ** n_bit - 1
    min_int = 0

    # get the scaling factor, zero point
    scaling_factor = (max_val - min_val).clamp(min=1e-5) / (max_int - min_int) # (len, 1)
    zero_point = (-torch.round(min_val / scaling_factor)).clamp_(0, max_int)
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    assert (not zero_point.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q =  torch.round( w / scaling_factor) + zero_point
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q - zero_point) * scaling_factor
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo quantization error: {max_error}")
    return w_f 

def pseudo_quantize_tensor_q40(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_abs_value = torch.amax(w.abs(), dim=-1, keepdim=True)
    max_int = (2 ** (n_bit - 1) - 1)
    min_int = - (2 ** (n_bit - 1) )

    # get the scaling factor, zero point
    scaling_factor = (max_abs_value).clamp(min=1e-5) / (min_int) # (len, 1)
    zero_point = torch.tensor(8).round()
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    assert (not zero_point.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q =  torch.round( w / scaling_factor)
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q) * scaling_factor
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo max quantization error: {max_error}")
    return w_f

def pseudo_quantize_tensor_q41(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_val = torch.amax(w, dim=-1, keepdim=True)
    min_val = torch.amin(w, dim=-1, keepdim=True)
    max_int = 2 ** n_bit - 1
    min_int = 0

    # get the scaling factor, zero point
    scaling_factor = (max_val - min_val).clamp(min=1e-5) / (max_int - min_int) # (len, 1)
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q = torch.round( 
            (w - min_val) / scaling_factor
        )
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q) * scaling_factor + min_val
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    # max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo quantization error: {max_error}")
    return w_f 

quantization_method_dict = {
    "q4z": pseudo_quantize_tensor_q4_zero_point,
    "q40": pseudo_quantize_tensor_q40,
    "q41": pseudo_quantize_tensor_q41
}

@torch.no_grad()
def pseudo_quantize_model_weight(model, w_bit, q_group_size, method):
    q_method = quantization_method_dict[method]
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            # print(f"Pseudo quantizing {n}")
            # m.weight.data = q_method(m.weight.data, w_bit, q_group_size)
            dq_weight = q_method(m.weight.data, w_bit, q_group_size)
            w_bias = None
            if hasattr(m, 'bias'):
                w_bias = m.bias
            custom_layer = QuantizedLinear(
                dq_weight,
                w_bias,
                8,
                4
            )
            del m.weight
            if hasattr(m, 'bias'):
                del m.bias

            model.set_submodule(
                n,
                custom_layer
            )
def quantize_and_evaluate(q_method):
    # load the tokenizer and the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )
    # Apply fake quantization
    pseudo_quantize_model_weight(model, 4, 32, q_method)
    # Evaluate the model
    model_perplexity = evaluate(model, testenc, tokenizer)
    model_size = get_model_size(model, data_width=4, group_size=32)
    print(f"\nmodel perplexity: {model_perplexity:.2f}")
    print(f"model size: {model_size/MiB:.2f} MiB")
    return model, tokenizer



## W41

In [ ]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q41")



Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  3.06it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:06<00:00,  6.50it/s]


model perplexity: 12.06
model size: 343.29 MiB


### GSM8k

In [8]:
evaluate_hf_model(model, tokenizer, batch_size=8)

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
2025-05-30:11:02:21 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:11:02:21 INFO     [evaluator:239] Using pre-initialized model
2025-05-30:11:02:34 INFO     [evaluator:286] gsm8k: Using gen_kwargs: {'until': ['Question:', '</s>', '<|im_end|>'], 'do_sample': False, 'temperature': 0.0}
2025-05-30:11:02:34 INFO     [api.task:434] Building contexts for gsm8k on rank 0...
100%|██████████| 1319/1319 [00:02<00:00, 467.07it/s]
2025-05-30:11:02:37 INFO     [evaluator:559] Running generate_until requests
Running generate_until requests: 100%|██████████| 1319/1

Results: {'gsm8k': {'alias': 'gsm8k', 'exact_match,strict-match': 0.8703563305534496, 'exact_match_stderr,strict-match': 0.009252657757825555, 'exact_match,flexible-extract': 0.8718726307808946, 'exact_match_stderr,flexible-extract': 0.009206398549980031}}


In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

> vLLM is not utilized here despite exhibiting way much faster evalution, because the linear layers are supeseded by custom layers, once it is saved in the disk, it could not be reloaded back. Plus, vLLM does not support hot-load our modified model 

In [ ]:

# !lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

## W4z

In [ ]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q4z")


Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  3.10it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:06<00:00,  6.49it/s]


model perplexity: 11.53
model size: 343.29 MiB


In [8]:
evaluate_hf_model(model, tokenizer, batch_size=14)

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
2025-05-30:11:53:17 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:11:53:17 INFO     [evaluator:239] Using pre-initialized model
2025-05-30:11:53:28 INFO     [evaluator:286] gsm8k: Using gen_kwargs: {'until': ['Question:', '</s>', '<|im_end|>'], 'do_sample': False, 'temperature': 0.0}
2025-05-30:11:53:28 INFO     [api.task:434] Building contexts for gsm8k on rank 0...
100%|██████████| 1319/1319 [00:02<00:00, 481.94it/s]
2025-05-30:11:53:31 INFO     [evaluator:559] Running generate_until requests
Running generate_until requests: 100%|██████████| 1319/1

Results: {'gsm8k': {'alias': 'gsm8k', 'exact_match,strict-match': 0.8529188779378317, 'exact_match_stderr,strict-match': 0.009756063660359894, 'exact_match,flexible-extract': 0.8582259287338894, 'exact_match_stderr,flexible-extract': 0.009608188527765585}}


In [ ]:

del tokenizer
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 
# !lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

## W40

In [ ]:
gc.collect()
torch.cuda.empty_cache()
model, tokenizer = quantize_and_evaluate("q40")


In [ ]:
evaluate_hf_model(model, tokenizer, batch_size=14)

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:


!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

remove perviously saved temporary model

In [1]:
!rm -rf tmp_quantized_model